# MP2: Control task in a continuous environment

In [ ]:
from collections import defaultdict
import numpy as np
import matplotlib.pyplot as plt
import torch
import pandas as pd
from tqdm import tqdm

from environments.acrobot_gym import Acrobot

In [ ]:
env = Acrobot()

## Simulation and plotting functions

In [ ]:
class AbstractAgent:
    """
    Abstract base class for agents interacting with the FrozenLake environment.
    Each agent must implement the `observe` method to update its internal state based on the observed transition,
    and the `act` method to choose an action given the current state.
    """
    def __init__(self, env: Acrobot):
        self.env = env

    def observe(self, state, action, reward, next_state, done):
        raise NotImplementedError

    def act(self, state):
        raise NotImplementedError
    

def run_agent(env: Acrobot, agent: AbstractAgent, num_episodes: int, print_every: int = 10):
    """
    Run the given agent in the environment for a specified number of episodes and collect statistics.
    """
    rewards = []
    episode_length = []
    for episode in tqdm(range(num_episodes)):
        state = env.reset()
        done = False
        steps = 0
        episode_reward = 0
        while not done:
            action = agent.act(state) # TODO
            next_state, reward = env.do_action(action)
            done = env.end
            steps += 1
            # TODO: update agent with the observed transition
            agent.observe(state, action, reward, next_state, done)
            state = next_state
            episode_reward += reward

        rewards.append(episode_reward)
        episode_length.append(steps)
        # TODO: log rewards and episode length
        if episode % print_every == 0:
            print(f"mean rewards since last print ({episode}) - {np.mean(rewards[-10:])}")
            print(f"mean steps since last print ({episode}) - {np.mean(episode_length[-10:])}")
        
        
    return rewards, episode_length

def gif_episode(env: Acrobot, agent: AbstractAgent, filename: str = "episode.gif"):
    """
    Run a single episode with the given agent and save the frames as a GIF.
    """
    state = env.reset()
    done = False
    while not done:
        action = agent.act(state)
        next_state, reward = env.do_action(action)
        done = env.end
        # no observe step for the agent
        state = next_state
    env.save_gif(filename)

def plot_rewards(rewards, window_size=10, savefig=False, filename="rewards_plot.png"):
    
    series = pd.Series(rewards)
    moving_avg = series.rolling(window=window_size, min_periods=1).mean()
    
    plt.figure(figsize=(10, 6))
    plt.plot(series, label='Episode Rewards', marker='o', linestyle='--', alpha=0.6, color='royalblue')

    plt.plot(moving_avg, label=f'Moving Average (Window {window_size})', linewidth=3, color='crimson')

    plt.title('Rewards per Episode', fontsize=14)
    plt.xlabel('Episode', fontsize=12)
    plt.ylabel('Reward', fontsize=12)
    plt.legend()
    plt.grid(True, linestyle=':', alpha=0.7)

    if savefig:
        plt.savefig(filename, dpi=300, bbox_inches='tight')
    plt.show()
    

def plot_success(rewards, window_size=10, savefig=False, filename="success_plot.png"):
    """
    Plot binary success and moving success rate.

    success_threshold:
        Episode is counted as success if total reward >= success_threshold.
        Adjust depending on how your Acrobot reward is defined.
    """
    successes = [1 if r > -500 else 0 for r in rewards]

    series = pd.Series(successes)
    moving_avg = series.rolling(window=window_size, min_periods=1).mean()

    plt.figure(figsize=(10, 6))
    plt.plot(series, label='Success (0/1)', marker='o', linestyle='--', alpha=0.5)
    plt.plot(moving_avg, label=f'Success Rate (Window {window_size})', linewidth=3)

    plt.title('Episode Success Rate')
    plt.xlabel('Episode')
    plt.ylabel('Success')
    plt.ylim(-0.05, 1.05)
    plt.legend()
    plt.grid(True, linestyle=':', alpha=0.7)
    if savefig:
        plt.savefig(filename, dpi=300, bbox_inches='tight')
    plt.show()

# additional functions can be added here
def output_data(rewards, episode_length, type, params, filename="agent_data.txt"):
    mean_reward = float(np.mean(rewards))
    success_percentage = float(np.mean([1 if r > -500 else 0 for r in rewards]) * 100)
    mean_episode_length = float(np.mean(episode_length))

    with open(filename, "a", encoding="utf-8") as f:
        f.write(f"Type: {type}\n")
        f.write(f"Parameters: {params}\n")
        f.write(f"Episodes: {len(rewards)}\n")
        f.write(f"Mean Reward: {mean_reward:.4f}\n")
        f.write(f"Final Mean Reward: {np.mean(rewards[-10:]):.4f}\n")
        f.write(f"Success Percentage: {success_percentage:.2f}%\n")
        f.write(f"Mean Episode Length: {mean_episode_length:.4f}\n")
        f.write("-" * 40 + "\n")

## Part 2: TD Learning

### Part 2.1: Random agent

Complete the tasks described in the project description and answer the following questions:

**Questions:**
1. How often does the random agent reach the target height?
2. What is the average reward collected by the random agent? Interpret this value in terms of episode duration.

**Your answers here** <br>
Disclaimer: We have set a random seed in the cell below that should make the plots reproducible
1. The agent reaches the target height around 2% of the time, see [here](Graphs/Random/random_agent_success.png). This can be attributed to lucky chance, as many correct random applications of the torque can lead to the agent reaching the target height without having a thorough strategy.
2. The average reward collected by the random agent is around -498, see [here](Graphs/Random/random_agent_rewards.png). This coincides with the average episode length as the reward is just the negative value of the length of episode. If an agent reached the target after 400 episodes, the reward would be -400. The episode length is capped at 500, so the worst reward is -500, which means that an average of -498 is very bad. This was, however, expected by the random agent.

In [ ]:
import random
random.seed(100)

In [ ]:
# Random agent
class RandomAgent(AbstractAgent):
    def observe(self, state, action, reward, next_state, done):
        pass  
    
    def act(self, state):
        # TODO: implement a random agent that selects actions uniformly at random
        return random.choice(env.available()) # [0, 1, 2]

random_ag = RandomAgent(env)

gif_episode(env, random_ag, filename="GIFs/Random/random_agent_episode.gif")

In [ ]:
random_rewards, random_episode_lengths = run_agent(env, random_ag, num_episodes=500, print_every=100)
plot_rewards(random_rewards, window_size=10, savefig=True, filename="Graphs/Random/random_agent_rewards.png")
plot_success(random_rewards, savefig=True, filename="Graphs/Random/random_agent_success.png")
output_data(random_rewards, random_episode_lengths, type="Random Agent", params="N/A", filename="output_data.txt")


In [ ]:
print(np.mean(random_rewards))
print(np.mean(random_episode_lengths))

### Part 2.2: Q-Learning with state discretization

Complete the tasks described in the project description and answer the following questions:

**Questions:**
1. For a given number of bins per dimension $n$, how many discrete states are there in total? How does this scale with the state dimension $d$?
2. What are the trade-offs of using a small number of bins? What are the trade-offs of using a large number of bins?
3. On average, which agent (random or Q-learning) will first reach the target height? Would this change with a different value of $\varepsilon$, a different Q-value initialization or a different reward function? Justify.
4. Does the Q-learning agent improve its performance over time? Does it manage to reach the goal in every episode eventually? Why or why not?
5. How does performance depend on n_bins? Which value gives the best results on this environment?

**Your answers here**
1. Say we have $n$ bins for $d$ dimensions. Then we will have $n^d$ states overall. This clearly scales exponentially if the number of dimensions $d$ increases.
2. A smaller number of bins would reduce the number of states overall. This means that the algorithm becomes less fine grained and therefore would end up in the same states more often than with more bins. It is tough to say whether this increases or decreases performance as it might be good to have fewer states to get more and better updates, however, also the granularity with more states could be prefered, that state the optimal action for more fine grained states.
3. The Q-Learning Agent will have the better performance and lower time to first success. The random agent has no strategy whatsoever and just randomly applies torque. The Q-Learning Agent is initialized optimistically (the Q-values are set to 0, which would be the perfect yet unrealistic reward) and therefore gets "disappointed" with every action especially early on. The utility of Q-Learning decreases with an increasing $\varepsilon$, if $\varepsilon = 1$ we have RandomAgent again. A changed reward function - such as having positive rewards and therefore initializing pessimistically - would still have the Q-Learning agent reach the target faster, but maybe the gap might close. 
4. Q-Learning generally does improve its performance over time. However, in this project setting this might happen only slowly because Q-Learning can only update the states one away. It also depends on the choice of bins, however, we can generally observe a slow increase in performance (decrease of negative average rewards). However, within the given limit of 500 steps per epsiode, even with the best setting, the bot does not reach its target every time.
5. The performance of the acrobot definitely does vary with the number of bins. The sweet spot we found was with 6 bins, see [here](Graphs\QLearn\qlearn_agent_success_6_bins.png), where it reaches the goal very often towards the end. 3 bins, see [here](Graphs\QLearn\qlearn_agent_success_3_bins.png) does not seem to be enough as there is little to no improvement across 500 episodes. For 9 bins the state space, see [here](Graphs\QLearn\qlearn_agent_success_9_bins.png) seems to be too sparse as the improvement only comes towards the end. If we increased the number of steps per episode, likely 9 bins would show improvement even if the 6 bins has levelled out.

In [ ]:
class StateDiscretizer:
    def __init__(self, env, n_bins):
        self.lower_bound, self.upper_bound = env.state_space
        self.n_bins = n_bins
        # your code here
        self.bins = []
        for low, high in zip(self.lower_bound, self.upper_bound):
            step = (high - low) / self.n_bins
            dim_bins = [low + step * i for i in range(self.n_bins + 1)]
            self.bins.append(dim_bins) 
    def __call__(self, state):
        # your code here
        states = []
        for idx, f in enumerate(state):
            states.append(np.digitize(f, self.bins[idx])-1)
        return states

# test the discretizer
state = env.reset()
discretizer = StateDiscretizer(env, n_bins=10)
discretized_state = discretizer(state)
print("Original state:", state)
print("Discretized state:", discretized_state)

In [ ]:
class QLearningAgent(AbstractAgent):
    def __init__(self, env: Acrobot, alpha=0.1, gamma=0.99, epsilon=0.1, n_bins=10):
        super().__init__(env)
        self.discretizer = StateDiscretizer(env, n_bins)
        # one Q-vector per discretized state, with one entry per action
        self.n_actions = env.n_actions
        self.Q = defaultdict(lambda: np.zeros(self.n_actions))

        self.alpha = alpha
        self.gamma = gamma
        self.epsilon = epsilon

    def observe(self, state, action, reward, next_state, done):
        # your code here
        current_tuple = tuple(self.discretizer(state))
        next_tuple = tuple(self.discretizer(next_state))

        if done:
            target = reward
        else:
            target = reward + self.gamma * np.max(self.Q[next_tuple])

        td_err = target - self.Q[current_tuple][action]

        self.Q[current_tuple][action] += self.alpha * td_err

    def act(self, state):
        # your code here
        s = tuple(self.discretizer(state))

        if np.random.rand() < self.epsilon:
            return np.random.choice(self.env.available())

        q_values = self.Q[s]
        best_actions = np.flatnonzero(q_values == np.max(q_values))
        return int(np.random.choice(best_actions))
    
# Analyze the Q-learning agent

for i in range(3, 10, 3):
    print(f"Running Q-learning agent with {i} bins per dimension...")
    qlearn_ag = QLearningAgent(env, n_bins=i)
    qlearn_rewards, qlearn_episode_lengths = run_agent(env, qlearn_ag, num_episodes=500, print_every=500)
    plot_rewards(qlearn_rewards, window_size=10, savefig=True, filename=f"Graphs/QLearn/qlearn_agent_rewards_{i}_bins.png")
    plot_success(qlearn_rewards, savefig=True, filename=f"Graphs/QLearn/qlearn_agent_success_{i}_bins.png")
    output_data(qlearn_rewards, qlearn_episode_lengths, type="Q-Learning Agent", params=f"n_bins={i}", filename="output_data.txt")
    gif_episode(env, qlearn_ag, filename=f"GIFs/QLearn/qlearn_agent_episode_{i}.gif")

### Part 2.3: Q-Learning with state discretization

Complete the tasks described in the project description and answer the following questions:

**Questions:**
1. Derive the partial derivative of $\mathcal{L}(w) = \frac{1}{2}\delta^2$ with respect to $w_{a,j}$, treating the target as a constant. Show your calculations and state the resulting update rule.
2. After observing a transition $(s_t,a_t,r_t,s_{t+1})$, which of the weights $w_{a,j}$ get updated and which stay unchanged?
3. What would differ between the bin-based Q-learning agent and the RBF agent in their treatment of a previously unseen state?
4. Does the RBF agent improve over time? Does it reach the goal consistently? Why or why not?
5. How does performance depend on n_centers_per_dim and $\alpha$? Which combination yields the best results?

**Your answers here**
1. If we write out the loss function we get 
$$\mathcal{L}(w) = \frac{1}{2} \left[ r + \gamma \max_{a'} \left( w_{a'}^\top \phi(s') \right) - \sum_{j=1}^{K} w_{a,j} \phi_j(s) \right]^2$$
Since the target ($r + \gamma \max_{a'} \left( w_{a'}^\top \phi(s') \right)$) is constant we know that its derivative is 0, while the derivative of the prediction ($\sum_{j=1}^{K} w_{a,j} \phi_j(s) $) can be computed but requires an indicator function and turns out to be $\mathbb{I}(a = a)\phi_j(s)$. We also have to use the chain rule for the outer $phi$ and the derivative of $\frac{1}{2} \delta^2 = \delta$. <br>
Then taking the partial derivative with respect to $w_{a,j}$, we get 
$$-\delta \mathbb{I}(a = a)\phi_j(s)$$
The indicator function means that only one weight will be updated at a time. Considering the update rule of 
$$w_{a,j} \leftarrow w_{a,j} - \alpha \frac{\partial}{\partial w_{a,j}} \mathcal{L}(w)$$
Plugging in the computed derivative gives us
$$w_{a,j} = w_{a,j} + \alpha \delta \phi_j(s)$$
Where the indicator function ensures that only this weight is updated
<br>
2. After we observe an action $a$, the algorithm updates the weight only corresponding to this action across all dimensions. No weights outside of that action are updated.
<br>
3. The soft assignment based on the rbf_features() function below allows better feedback for the algorithm. Previously unseen states are handled with 0-Q-values in the Q-Learnig algorithm, while in RBF can compute an average of nearby centers to come up with a better than standard estimate. This is done using the $\phi$ parameter.
<br>
4. The RBF agent clearly improves over time. In most tried combinations of centers and alphas, it is able to reach the goal every time even much before the 500 episode horizon. This is likely due to the better assignment of centers, which happens in a soft matter rather than the hard assignment seen in Q-Learning. Updates have an impact on more possible states due to this assigment. Additionally, RBF is also more efficient.
<br>
5. Yes, performance depends on these parameters as well. We have observed the best performance with $\alpha=0.05$ and $n=4$ for the centers, see [here](Graphs/RBF/rbf_agent_rewards_alpha0.05_centers4.png). This combination leads to the bot reaching its goal almost always for each of the last 400 episodes with average reward being at -201 for the last 10 episodes observed. In almost all of the tried combinations of the two parameters (4-6 centers, $\alpha \in {0.001, 0.005, 0.001, 0.005, 0.01}$) the success percentage is >60% with very high success rate over the last 250 episodes. Just $\alpha=0.001$ seems to be too low because performance drops. However, the rewards seem to vary, with performance declining in both direction of the aforementioned sweet spot of $\alpha=0.05$ and $n=4$ for the centers.

In [ ]:
def rbf_features(state, centers, sigma):
    """
    Compute the RBF features for a given state, given the centers and sigma.
    Arguments:
        - state: the input state, a vector of shape (state_dim,)
        - centers: a np array of shape (n_centers, state_dim)
        - sigma: a vector of shape (state_dim,)
    """
    diff = centers - state
    normalized_diff = diff / sigma
    
    sq_dist = np.sum(normalized_diff**2, axis=1)
    return np.exp(-0.5 * sq_dist)
    


In [ ]:
class RBFAgent(AbstractAgent):
    """
    Agent using Radial Basis Function (RBF) features to approximate the Q-function.
    """
    def __init__(self, env: Acrobot, alpha: float = 0.1, gamma: float = 0.9, epsilon_exploration: float = 0.1, n_centers_per_dim: int = 5):
        super().__init__(env)
        # your code here
        self.lower_bound, self.upper_bound = env.state_space
        axes = np.array([
            np.linspace(self.lower_bound[d], self.upper_bound[d], n_centers_per_dim * 2 + 1)[1::2]
            for d in range(len(self.lower_bound))
        ])
        import itertools
        self.centers = np.array(list(itertools.product(*axes)))
        self.sigmas = 0.5 * np.diff(axes[:, :2], axis=1).flatten()
        self.Q_weights = np.zeros((env.n_actions, self.centers.shape[0]))
        self.alpha = alpha
        self.gamma = gamma
        self.epsilon_exploration = epsilon_exploration

    def observe(self, state: int, action: int, reward: float, next_state: int, done: bool):
        # your code here
        phi = rbf_features(state, self.centers, self.sigmas)
        q_sa = self.Q_weights[action] @ phi

        if done:
            target = reward
        else:
            phi_next = rbf_features(next_state, self.centers, self.sigmas)
            q_next_all = self.Q_weights @ phi_next
            target = reward + self.gamma * np.max(q_next_all)

        td_error = target - q_sa
        self.Q_weights[action] += self.alpha * td_error * phi

    def act(self, state: int) -> int:
        # your code here
        if np.random.rand() < self.epsilon_exploration:
            return np.random.choice(self.env.available())
        phi = rbf_features(state, self.centers, self.sigmas)
        q_values = self.Q_weights @ phi
        best_actions = np.flatnonzero(q_values == np.max(q_values))
        return int(np.random.choice(best_actions))
    
# Analyze the RBF agent
values = [1e-3, 5e-3, 1e-2, 5e-2, 1e-1]
for a in values:
    for c in range(4, 7, 1):
        print(f"Running RBF-learning agent with {c} centers and Alpha={a}...")
        rbfAgent = RBFAgent(env, alpha=a, n_centers_per_dim=c)
        rbf_rewards, rbf_episode_lengths = run_agent(env, rbfAgent, num_episodes=500, print_every=500)
        plot_rewards(rbf_rewards, window_size=10, savefig=True, filename=f"Graphs/RBF/rbf_agent_rewards_alpha{a}_centers{c}.png")
        plot_success(rbf_rewards, savefig=True, filename=f"Graphs/RBF/rbf_agent_success_alpha{a}_centers{c}.png")
        output_data(rbf_rewards, rbf_episode_lengths, type="RBF Agent", params=f"alpha={a}, n_centers_per_dim={c}", filename="output_data.txt")
        gif_episode(env, rbfAgent, filename=f"GIFs/RBF/rbf_agent_episode{str(a).split('.')[1]}_{c}.gif")

## Part 3: Policy gradient

### part 3.1: REINFORCE with a linear softmax policy

Complete the tasks described in the project description and answer the following questions:

**Questions:**
1. What gradient expression did you derive? Show your calculations.
2. What are the main differences between REINFORCE and Q-learning with RBFs in terms of (i) what they learn, (ii) how they update, and (iii) the kind of policy they produce?
3. Could we use an $\varepsilon$-greedy policy with REINFORCE? Why or why not?
4. Does the agent improve over time? Comment on your plots.

**Your answers here**
1. We start by taking the logarithm of the softmax function, we get $$\ln \pi_\theta(a | s) = \theta_a^\top \phi(s) - \ln \left( \sum_{a'} \exp(\theta_{a'}^\top \phi(s)) \right)$$ 
Then we take the gradient step by step, and observe that the first term changes as following 
$$\nabla_{\theta_k} \theta_a^\top \phi(s) = \mathbb{I}(k = a_t) \phi(s)$$

The derivative of the second part is the following 
$$\nabla_{\theta_k} \ln \left( \sum_{a'} \exp(\theta_{a'}^\top \phi(s_t)) \right) = \frac{1}{\sum_{a'} \exp(\theta_{a'}^\top \phi(s_t))} \cdot \nabla_{\theta_k} \sum_{a'} \exp(\theta_{a'}^\top \phi(s_t))$$

And since the sum only involves one term with $\theta_k$, it collapses to the following

$$\frac{1}{\sum_{a'} \exp(\theta_{a'}^\top \phi(s_t))} \cdot \exp(\theta_k^\top \phi(s_t))\phi(s_t) = \pi_\theta(k|s_t)\phi(s_t)$$

Then the whole expression becomes

$$\nabla \ln \pi_{\theta}(a_t | s_t, \theta) = (\mathbb{I}(k = a_t) - \pi_\theta(k|s_t)) \cdot \phi(s_t)$$

2. (i) Q-learning learns an action-value function mapping state-action pairs to scalars. REINFORCE learns a policy directly, where weights represent probabilities over actions.

    (ii) Q-learning updates every step using a bootstrapped TD target. REINFORCE updates after a full episode using real Monte Carlo returns.

    (iii) Q-learning produces a deterministic policy with randomness injected via $\varepsilon$-greedy.
REINFORCE produces an inherently stochastic policy, sampling actions according to learned probabilities.

3. No. REINFORCE is an on-policy method, its gradient estimate requires actions to be sampled from the policy being updated. $\varepsilon$-greedy introduces random actions that would not correspond to policy, which would bias the gradient and corrupt the training.

4. The agent does not improve over time. Variance in returns is too high for stable learning and performance is heavily dependent on initialization. Across all parameters we tried even the best runs are only comparable to random agent.

In [ ]:
class REINFORCEAgent(AbstractAgent):
    """
    Bounds for dim_pints include endpoints, half sigmas 
    """        
    def __init__(self, env: Acrobot, alpha: float = 0.1, gamma: float = 0.9, n_centers_per_dim: int = 5):
        super().__init__(env)
        # your code here
        self.alpha = alpha
        self.gamma = gamma
        lower_bound, upper_bound = env.state_space
        dim_points = [
            np.linspace(lower_bound[d], upper_bound[d], n_centers_per_dim)
            for d in range(len(lower_bound))
        ]
        import itertools
        self.centers = np.array(list(itertools.product(*dim_points)))
        self.sigmas = np.array([(p.max() - p.min()) / (2 * (len(p) - 1)) for p in dim_points]) #works if n_centers_per_dim >= 2
        
        self.policy_weights = np.zeros((env.n_actions, self.centers.shape[0]))        
        self.episode_memory = []  # to store (state, action, reward) for the current episode


    def observe(self, state: int, action: int, reward: float, next_state: int, done: bool):
        # your code here
        self.episode_memory.append((state, action, reward))

        if not done: #weights updated only after full episode
            return

        # Compute returns G_t
        T = len(self.episode_memory)
        rewards = np.array([r for (_, _, r) in self.episode_memory])

        for t, (s, a, _) in enumerate(self.episode_memory):
            G = (rewards[t:T] * self.gamma ** np.arange(T - t)).sum()
            
            phi = rbf_features(s, self.centers, self.sigmas)          # (K,)
            scores = self.policy_weights @ phi                        # (n_actions,)
            exp_scores = np.exp(scores)
            probs = exp_scores / np.sum(exp_scores) # π(.|s)
            
            # gradient
            # update policy_weights
            
            #aproach 1
            for a_prime in range(self.env.n_actions):
                indicator = 1.0 if a_prime == a else 0.0
                grad_log_policy = (indicator - probs[a_prime]) * phi

                self.policy_weights[a_prime] += self.alpha * G * grad_log_policy
            
            # aproach 2
            # grad = - np.outer(phi, probs)
            # grad[:, a] += phi
            # self.policy_weights += self.alpha * G * grad.T

        self.episode_memory = []
            
    def act(self, state: int) -> int:
        phi = rbf_features(state, self.centers, self.sigmas)          # (K,)
        scores = self.policy_weights @ phi                            # (n_actions,)
        # scores = scores - np.max(scores)                      #acc to gpt without scaling it could overflow
        exp_scores = np.exp(scores)
        probs = exp_scores / np.sum(exp_scores)
        action = int(np.random.choice(self.env.n_actions, p=probs)) # sample action according to learned probabilities (softmax policy)
        return action




# Analyze the REINFORCE agent
# ...

In [ ]:
# agent = REINFORCEAgent(env, alpha = 0.005, n_centers_per_dim = 4)

# rewards, episode_lengths = run_agent(env, agent, num_episodes=100, print_every=10)

# plot_rewards(rewards)
# plot_success(rewards)
# gif_episode(env, agent, filename="joke.gif")


# Analyze the REINFORCE agent
values = [1e-3, 5e-3, 1e-2, 5e-2, 1e-1]
centers_range = range(4, 7)

for a in values:
    for c in centers_range:
        print(f"Running REINFORCE agent with alpha={a}, centers={c}...")
        agent = REINFORCEAgent(env, alpha=a, n_centers_per_dim=c)

        rewards, episode_lengths = run_agent(env, agent, num_episodes=500, print_every=500)

        plot_rewards(rewards, window_size=10, savefig=True, filename=f"Graphs/REINFORCE/reinforce_rewards_alpha{a}_centers{c}.png")
        plot_success(rewards, savefig=True, filename=f"Graphs/REINFORCE/reinforce_success_alpha{a}_centers{c}.png")

        output_data(rewards, episode_lengths, type="REINFORCE Agent", params=f"alpha={a}, n_centers_per_dim={c}", filename="output_data.txt")

        gif_episode(env, agent, filename=f"GIFs/REINFORCE/reinforce_episode_{str(a).replace('.', '')}_{c}.gif")

### part 3.2: REINFORCE with a Multi-Layer Perceptron (MLP)

Complete the tasks described in the project description and answer the following questions:

**Questions:**
1. Compared to the linear softmax policy, what has changed architecturally, and why does this make the policy more expressive?
2. Compare the performance of linear and MLP REINFORCE agents. Is there a significant improvement?
3. For which value of the learning rate do you find the best results?

**Your answers here**

1. The linear softmax policy used hand crafted RBF features and representation of the state was fixed before learning. The MLP adds a learnable nonlinear layer that lets the agent learn best representation on its own.

2. Depending on initialization and learning rate, the MLP agent starts learning but eventually 
sabotages itself and loses all progress. Unlike the linear agent which arguably showed no learning at 
all, some MLP runs do achieve success under 250 steps before forgetting it.

3. lr=0.0001 showed some learning but was sabotaged slowly after, but it started learning again before termination achieving ~20% success. lr=0.005 improved fast, collapsed, 
recovered, then collapsed again. If 0.005 retained its peak performance it would be the best, 
but since 0.0001 degrades more gradually it is the more stable choice. Both extremes, lr=0.01 and lr=0.0005, achieved 0 successes (worse than random). Overall 
training is highly stochastic across all learning rates.

In [ ]:
# use pytorch to implement a MLP policy network and train it with REINFORCE
class REINFORCEAgentMLP(AbstractAgent):
    def __init__(self, env: Acrobot, alpha: float = 0.1, gamma: float = 0.9, hidden_size: int = 64):
        super().__init__(env)
        
        # your code here
        self.gamma = gamma
        self.alpha = alpha
        self.n_actions = env.n_actions
        self.state_dim = env.state_dim
        # simple MLP with one hidden layer
        self.policy_mlp = torch.nn.Sequential(
            torch.nn.Linear(self.state_dim, hidden_size),
            torch.nn.ReLU(),
            torch.nn.Linear(hidden_size, self.n_actions)
        )
        self.optimizer = torch.optim.Adam(self.policy_mlp.parameters(), lr=self.alpha)
        self.episode_memory = []  # to store (state, action, reward) for the current episode

    def _compute_probs(self, state):
        """
        
        """
        state_tensor = torch.FloatTensor(state)
        logits = self.policy_mlp(state_tensor)
        exp_logits = torch.exp(logits - torch.max(logits))  # for numerical stability
        probs = exp_logits / (torch.sum(exp_logits) + 1e-8)  # add small value to avoid division by zero
        return probs
    
    def _compute_loss_term(self, state, action, G):
        probs = self._compute_probs(state)
        log_prob = torch.log(probs[action])
        return -log_prob * G
    
    def _update_parameters(self, sum_of_loss_terms):
        self.optimizer.zero_grad()
        sum_of_loss_terms.backward()
        self.optimizer.step()
    
    def act(self, state):
        # your code here (use _compute_probs)
        probs = self._compute_probs(state)
        action = torch.multinomial(probs, 1).item()        
        return action
    
    
    def observe(self, state, action, reward, next_state, done):
        # your code here
        # hint: compute G for each loss term, then use _compute_loss_term and _update_parameters
        self.episode_memory.append((state, action, reward))
        
        if not done: #weights updated only after full episode
            return
        
        # Compute returns G_t
        T = len(self.episode_memory)
        rewards = np.array([r for (_, _, r) in self.episode_memory])
        
        loss_terms = []

        for t, (s, a, _) in enumerate(self.episode_memory):
            G = (rewards[t:T] * self.gamma ** np.arange(T - t)).sum()
            G = float(G) # convert numpy scalar to Python float to avoid issues with PyTorch ops (GPT recommendation)
            loss_terms.append(self._compute_loss_term(s, a, G))
        
        self._update_parameters(torch.stack(loss_terms).sum())
        self.episode_memory = []

In [ ]:
# agent = REINFORCEAgentMLP(env, alpha=1e-3)

# rewards, episode_lengths = run_agent(env, agent, num_episodes=100, print_every=10)

# plot_rewards(rewards)
# plot_success(rewards)
# gif_episode(env, agent, filename="joke.gif")

# # Analyze the REINFORCE MLP agent
values = [5e-4, 1e-3, 5e-3, 1e-2, 5e-2, 1e-1]

for a in values:
    print(f"Running REINFORCE MLP agent with alpha={a}...")
    mlpAgent = REINFORCEAgentMLP(env, alpha=a)
    mlp_rewards, mlp_episode_lengths = run_agent(env, mlpAgent, num_episodes=500, print_every=500)
    plot_rewards(mlp_rewards, window_size=10, savefig=True, filename=f"Graphs/REINFORCE_MLP/mlp_rewards_alpha{a}.png")
    plot_success(mlp_rewards, savefig=True, filename=f"Graphs/REINFORCE_MLP/mlp_success_alpha{a}.png")
    output_data(mlp_rewards, mlp_episode_lengths, type="REINFORCE MLP Agent", params=f"alpha={a}", filename="output_data.txt")
    gif_episode(env, mlpAgent, filename=f"GIFs/REINFORCE_MLP/mlp_episode_alpha{str(a).replace('.', '_')}.gif")

## part 4: Actor-Critic

Complete the tasks described in the project description and answer the following questions:

**Questions:**
1. What are the key differences between REINFORCE and the actor-critic algorithm in terms
of (i) when updates are performed, (ii) what signal is used to update the policy, and (iii)
variance of the gradient estimate?  
2. How does the actor-critic agent’s learning curve compare to REINFORCE? Discuss differ-
ences in speed, stability, and final performance.  
3. Which values for the actor’s learning rate and the critic’s learning rate give you the best
performance?  
4. Visualize one episode of the trained actor-critic agent. Describe the strategy it has learned.
Does the behavior match your intuition about how a human would solve the task?   

**Your answers here**
1. (i) REINFORCE updates after a full episode, implemented Actor-Critic updates every step.

    (ii) REINFORCE uses full Monte Carlo returns. Actor-Critic uses a TD advantage 
$\mathcal{A}_t = r_t + \gamma V(s_{t+1}) - V(s_t)$, comparing each step against 
the critic's baseline rather than waiting for the full return.

    (iii) TD advantage has lower variance than Monte Carlo returns, preventing the 
detraining and progress loss seen in REINFORCE.

2. Actor-Critic learns faster and is more stable than REINFORCE and it retains learning progress but it is dependent on actor critic learning rate. Final performance is significantly better, with a good setup achieving ~100 steps and near 100% success rate, compared to REINFORCE which is unreliable and performs worse. Higher learning rates speed up learning but introduce more instability.


3. Best results with α_actor=0.0001, α_critic=0.0005 (100% success, final reward ~−100). 
It learns slower but retains 100% once reached. Higher learning rates achieve similar 
performance with occasional drops, except α_actor=0.001, α_critic=0.005 which learned 
faster and also retained it. Actor learning rate must be smaller than the critics, otherwise the model collapses to 0% success, losing all learned performance.

4. Visualized below. The arm swings back and forth until it builds enough momentum to reach the top, which matches intuition. The strategy is inefficient with redundant swinging, but a 
person manually controlling the joints would likely perform similarly at first.

In [ ]:
class ActorCriticAgent(AbstractAgent):
    def __init__(self, env: Acrobot, alpha_actor: float = 0.1, alpha_critic: float = 0.1, gamma: float = 0.9, hidden_size: int = 64):
        super().__init__(env)
        # your code here
        self.gamma = gamma
        self.n_actions = env.n_actions
        self.state_dim = env.state_dim
        # simple MLP for the policy (actor)
        self.policy_mlp = torch.nn.Sequential(
            torch.nn.Linear(self.state_dim, hidden_size),
            torch.nn.ReLU(),
            torch.nn.Linear(hidden_size, self.n_actions)
        )
        # simple MLP for the value function (critic)
        self.value_mlp = torch.nn.Sequential(
            torch.nn.Linear(self.state_dim, hidden_size),
            torch.nn.ReLU(),
            torch.nn.Linear(hidden_size, 1)
        )
        self.optimizer_actor = torch.optim.Adam(self.policy_mlp.parameters(), lr=alpha_actor)
        self.optimizer_critic = torch.optim.Adam(self.value_mlp.parameters(), lr=alpha_critic)


    def _compute_probs(self, state):
        state_tensor = torch.FloatTensor(state)
        logits = self.policy_mlp(state_tensor)
        exp_logits = torch.exp(logits - torch.max(logits))  # for numerical stability
        probs = exp_logits / torch.sum(exp_logits) + 1e-8  # add small value to avoid division by zero
        return probs
    
    def _compute_value(self, state):
        state_tensor = torch.FloatTensor(state)
        value = self.value_mlp(state_tensor)
        return value.squeeze()  # remove extra dimension
    
    def _compute_advantage(self, state, reward, next_state, done):
        # your code here (compute advantage using the critic)
        # hint: use _compute_value
        v_state = self._compute_value(state)
        with torch.no_grad():
            v_next_state = self._compute_value(next_state) #fix the target as in 2.3
        
        if done:
            target = reward
        else:
            target = reward + self.gamma * v_next_state
        
        advantage = target - v_state
        return advantage

    def _compute_actor_loss(self, state, action, reward, next_state, done):
        with torch.no_grad():
            advantage = self._compute_advantage(state, reward, next_state, done)
        probs = self._compute_probs(state)
        log_prob = torch.log(probs[action])

        return -log_prob * advantage
    
    def _compute_critic_loss(self, state, reward, next_state, done):
        advantage = self._compute_advantage(state, reward, next_state, done)
        return advantage.pow(2)  # MSE loss for the critic
    
    def _update_parameters(self, actor_loss, critic_loss):
        # update actor
        self.optimizer_actor.zero_grad()
        actor_loss.backward()
        self.optimizer_actor.step()
        # update critic
        self.optimizer_critic.zero_grad()
        critic_loss.backward()
        self.optimizer_critic.step()

    def act(self, state):
        # your code here (use _compute_probs)
        probs = self._compute_probs(state)
        action = torch.multinomial(probs, 1).item()        
        return action
    
    def observe(self, state, action, reward, next_state, done):
        # your code here
        # hint: compute advantage, then compute actor and critic loss, then update parameters
        critic_loss = self._compute_critic_loss(state, reward, next_state, done)
        actor_loss = self._compute_actor_loss(state, action, reward, next_state, done)
        self._update_parameters(actor_loss, critic_loss)
    

In [ ]:
# agent = ActorCriticAgent(env, alpha_actor=1e-3, alpha_critic=1e-3)

# rewards, episode_lengths = run_agent(env, agent, num_episodes=100, print_every=10)

# plot_rewards(rewards)
# plot_success(rewards)
# gif_episode(env, agent, filename="joke.gif")

# # Analyze the Actor-Critic agent
actor_values = [5e-4, 1e-3, 5e-3, 1e-2, 5e-2, 1e-1]
critic_values = [5e-4, 1e-3, 5e-3, 1e-2, 5e-2, 1e-1]

for a_actor in actor_values:
    for a_critic in critic_values:
        print(f"Running Actor-Critic agent with alpha_actor={a_actor}, alpha_critic={a_critic}...")
        acAgent = ActorCriticAgent(env, alpha_actor=a_actor, alpha_critic=a_critic)
        ac_rewards, ac_episode_lengths = run_agent(env, acAgent, num_episodes=500, print_every=500)
        plot_rewards(ac_rewards, window_size=10, savefig=True, filename=f"Graphs/ActorCritic/ac_rewards_actor{a_actor}_critic{a_critic}.png")
        plot_success(ac_rewards, savefig=True, filename=f"Graphs/ActorCritic/ac_success_actor{a_actor}_critic{a_critic}.png")
        output_data(ac_rewards, ac_episode_lengths, type="Actor-Critic Agent", params=f"alpha_actor={a_actor}, alpha_critic={a_critic}", filename="output_data.txt")
        gif_episode(env, acAgent, filename=f"GIFs/ActorCritic/ac_episode_actor{str(a_actor).replace('.', '_')}_critic{str(a_critic).replace('.', '_')}.gif")

In [ ]:
from IPython.display import Image, display, clear_output
import time

for _ in range(3):
    display(Image(r'.\GIFs\ActorCritic\ac_episode_actor0_001_critic0_005.gif'))
    time.sleep(4)
    clear_output(wait=True)